In [3]:
import sys
sys.path.append('/home/igavier_umass_edu/Documents/Neuromorphic-IMU/snn/')
from har_snn import *

from neurobench.models import TorchModel, SNNTorchModel
from neurobench.benchmarks import Benchmark
from neurobench.metrics.static import Footprint, ConnectionSparsity
from neurobench.metrics.workload import ActivationSparsity, SynapticOperations

In [13]:
tc = 8.0

kwargs = {}
kwargs.update([('shiftSyn', 1), ('shiftMem', 1)])
kwargs['timeResolution'] = 1.0 / 64 * tc
kwargs['tausFactor'] = 1.0 / tc

kwargs['reset'] = False

model = createModel(
    'CNNNetSNN',
    inputSize=15,
    outputSize=8,
    hiddenSizes=(80,80,80),
    device=torch.device('cpu'),
    sampleFreq=64,
    **kwargs
)#.to_torch()

model = TorchModel(model)
train_loader, val_loader, test_loader = createLoaders(
    '/work/pi_sunghoonlee_umass_edu/Ignacio/realworld/data/eventsNIMU/windows/',
    datasetName='Realworld',
    subjectSplits=(80,10,10),
    batchSize=8,
    randomSeed=1,
    balance=False,
    sampleFreq=64,
    rectifySpikes=False,
    polarityBichannel=False,
    timeCompress=tc,
    timeResolution=1.0 / 64 * tc,
    systemType='downsampled',
    quantizeSpikes=False
)

Subject splits: [array([ 1, 10,  7,  9, 13,  4,  5,  8,  0,  2, 12]), array([11,  6]), array([3])]
Subject splits: [array([ 1, 10,  7,  9, 13,  4,  5,  8,  0,  2, 12]), array([11,  6]), array([3])]
Subject splits: [array([ 1, 10,  7,  9, 13,  4,  5,  8,  0,  2, 12]), array([11,  6]), array([3])]


In [14]:
static_metrics = [Footprint, ConnectionSparsity]
data_metrics = [ActivationSparsity, SynapticOperations]

benchmark = Benchmark(
   model,
   test_loader,
   [],
   [],
   [static_metrics, data_metrics]
)

In [15]:
results = benchmark.run()
print(results)

Running benchmark


100%|██████████| 105/105 [00:17<00:00,  5.84it/s]

{'Footprint': 397976, 'ConnectionSparsity': 0.0, 'ActivationSparsity': 0.0, 'SynapticOperations': {'Effective_MACs': 6388618.934289128, 'Effective_ACs': 42479475.15412186, 'Dense': 191353600.0}}
